# Fase 1 — Predicción de retrasos en la llegada de vuelos

Modelos y Simulación de Sistemas I — 2026-II · Grupo B


## 0. Configuración


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Semilla global del proyecto: garantiza reproducibilidad en splits y estimadores
RANDOM_STATE = 42


In [ ]:
# Ruta relativa
df = pd.read_csv("data/flight_data_2024_sample.csv")


## 1. Introducción


### 1.1 Descripción del problema

El transporte aéreo comercial funciona como una red fuertemente acoplada: cada aeronave
encadena varios vuelos a lo largo del día, las tripulaciones tienen jornadas reguladas y
los aeropuertos operan con franjas horarias asignadas con poco margen. En un sistema así,
un retraso rara vez se queda donde nace. Un vuelo que sale tarde de su primer origen
arrastra el desfase a los tramos siguientes de la misma aeronave, puede dejar a una
tripulación fuera de su límite legal de servicio y hace que pasajeros con conexión pierdan
su enlace. Lo que empieza como una demora local se propaga por la red durante el resto de
la jornada.

Las consecuencias son a la vez logísticas y económicas. Para la aerolínea, un retraso
significa combustible adicional, reacomodación de pasajeros, compensaciones, personal en
sobretiempo y, sobre todo, una programación que hay que rehacer sobre la marcha. Para los
aeropuertos implica congestión de posiciones y de personal en tierra en momentos que no
estaban previstos. Y para el pasajero se traduce en tiempo perdido, conexiones falladas y
planes rotos. Por eso los organismos reguladores estadounidenses —la FAA y el Bureau of
Transportation Statistics— llevan décadas midiendo y publicando la puntualidad de cada
vuelo doméstico: es un indicador de desempeño del sector, no un dato anecdótico.

Frente a este panorama, poder **anticipar** el retraso de un vuelo antes de que despegue
tiene un valor operativo claro. Permite avisar a los pasajeros con tiempo, replanificar
conexiones ajustadas, redistribuir recursos en tierra hacia los vuelos que probablemente
los necesitarán y tomar decisiones de programación con una expectativa cuantificada en
lugar de con una intuición. Ese es el problema que aborda este trabajo.


### 1.2 Objetivo del modelo

El objetivo es construir un modelo capaz de estimar cuántos minutos de retraso acumulará
un vuelo en su llegada, utilizando **exclusivamente información conocida antes del
despegue**: la aerolínea operadora, los aeropuertos de origen y destino, los horarios
programados de salida y llegada, la duración y la distancia previstas, y la posición del
vuelo en el calendario (mes y día de la semana).

Esa restricción es deliberada y es, de hecho, lo que decide si el modelo sirve para algo.
El conjunto de datos incluye campos que describen el vuelo con enorme precisión —el
retraso en la salida, los tiempos de rodaje, las horas reales de despegue y aterrizaje—
pero todos ellos se conocen *durante o después* del vuelo. Un modelo que los utilizara
alcanzaría un desempeño excelente en la evaluación y sería inútil en producción, porque en
el momento en que la predicción tendría valor —horas antes del despegue, cuando todavía se
puede reprogramar una conexión o avisar a un pasajero— esos datos sencillamente no
existen. Usarlos constituiría **fuga de información**, y evitarla es la restricción central
de este trabajo.

Dicho de otro modo: preferimos un modelo honesto y menos preciso, cuyas predicciones puedan
producirse en el instante en que se necesitan, antes que un modelo aparentemente
sobresaliente que solo funciona cuando la respuesta ya se conoce. Las variables excluidas
se enumeran y se justifican una por una en la sección 4, y las medidas concretas adoptadas
para prevenir la fuga se documentan en la sección 6.


### 1.3 Tipo de problema

El trabajo se plantea como un problema de **regresión supervisada**. Es supervisado porque
cada vuelo histórico viene acompañado de su retraso real observado, que actúa como
etiqueta con la que el modelo aprende; y es de regresión porque la cantidad que se busca
predecir, `arr_delay`, es continua y se mide en minutos.

Existía una alternativa razonable: convertir el problema en una **clasificación binaria**
definiendo `is_delayed = (arr_delay > 15).astype(int)`, donde 15 minutos es el umbral con
el que la FAA y el BTS consideran oficialmente que un vuelo llegó tarde. Esa formulación
es más sencilla de evaluar y suele producir métricas más vistosas.

Se opta por la regresión por dos razones. La primera es informativa: la clasificación
binaria descarta la magnitud del retraso, y para las decisiones que motivan el modelo esa
magnitud es justamente lo que importa — un retraso de 20 minutos y uno de tres horas caen
en la misma clase pero tienen consecuencias operativas incomparables. La segunda es
metodológica: la salida continua es más general, ya que a partir de una predicción en
minutos siempre se puede derivar la clasificación aplicando el umbral, mientras que el
camino inverso no existe. Se asume a cambio un problema más exigente, con métricas que
previsiblemente serán modestas; ese resultado se interpreta con honestidad en la
sección 10 en lugar de esquivarse.


### 1.4 Variable objetivo

La variable objetivo es **`arr_delay`**: la diferencia, **en minutos**, entre la hora real
de llegada del vuelo y la hora de llegada programada en el itinerario.

Su signo tiene significado propio y conviene tenerlo presente desde el principio: un valor
positivo indica que el vuelo llegó tarde, un valor negativo indica que llegó adelantado —
algo frecuente, porque las aerolíneas incorporan holgura en los tiempos de bloque
publicados — y un valor cercano a cero indica que llegó en hora. Por eso el objetivo no es
una magnitud estrictamente positiva, y cualquier transformación que asuma lo contrario
(un logaritmo, por ejemplo) requeriría un tratamiento previo del rango negativo.

La distribución de `arr_delay`, su asimetría y sus valores atípicos se examinan con datos
en la sección 3.


### 1.5 Fuente de los datos

Los datos provienen del conjunto **Flight Delay Dataset 2024**, publicado por Hrishit Patil
en Kaggle:

> https://www.kaggle.com/datasets/hrishitpatil/flight-data-2024

Ese conjunto no es una recolección propia del autor, sino una consolidación de los
registros públicos del **Bureau of Transportation Statistics (BTS)** del Departamento de
Transporte de los Estados Unidos, concretamente de la base *TranStats On-Time
Performance*, en la que las aerolíneas comerciales están obligadas por regulación a
reportar mes a mes los horarios programados y reales de cada vuelo doméstico. Que el
origen sea un registro regulatorio obligatorio, y no una muestra voluntaria, es una
garantía razonable de cobertura y de consistencia en la definición de los campos.

Por su volumen, el archivo completo no se versiona en este repositorio. El trabajo se
realiza sobre la muestra `data/flight_data_2024_sample.csv`, cuyas dimensiones exactas se
verifican con código en la sección 2. Las instrucciones para obtener los datos se
documentan en el `README.md` de esta fase.


## 2. Descripción del conjunto de datos


### 2.1 Dimensiones del conjunto

Para el desarrollo de esta fase trabajamos sobre la muestra `flight_data_2024_sample.csv`. Como punto de partida, verificamos las dimensiones del conjunto y definimos tanto las variables predictoras preseleccionadas como la variable objetivo, dejándolas como constantes que reutilizaremos a lo largo de todo el notebook.

In [ ]:
PREDICTORAS = [
    "op_unique_carrier", "origin", "dest",
    "crs_dep_time", "crs_arr_time", "crs_elapsed_time",
    "distance", "month", "day_of_week",
]
OBJETIVO = "arr_delay"

n_obs, n_cols = df.shape
print(f"Observaciones:        {n_obs:,}")
print(f"Columnas totales:     {n_cols}")
print(f"Predictoras usadas:   {len(PREDICTORAS)}")

MIN_OBS = 1000
cumple = n_obs >= MIN_OBS
print(f"\n¿Cumple el mínimo de {MIN_OBS:,} observaciones? "
      f"{'SÍ' if cumple else 'NO'}  ({n_obs:,} >= {MIN_OBS:,})")

El dataset cargado cuenta con **10 000 observaciones y 35 columnas**. De este total, seleccionamos **9 variables predictoras**, cumpliendo con los requisitos establecidos para la entrega (un mínimo de 1 000 registros y al menos 5 variables explicativas).

### 2.2 Significado de la variable objetivo

Nuestra variable de interés es **`arr_delay`**, que representa la diferencia (en minutos) entre la hora real en que aterriza el vuelo y la hora que estaba programada en su itinerario original.

Un aspecto fundamental a tener en cuenta es el sentido de su signo:
- **Valores positivos:** representan retrasos en la llegada.
- **Valores negativos:** indican que el vuelo **llegó antes de lo previsto**.
- **Valores cercanos a cero:** corresponden a vuelos puntuales.

En aviación comercial, que un vuelo tenga retraso negativo no es una anomalía de los datos ni un error de medición. De hecho, como veremos en las estadísticas descriptivas, **más del 60 % de los vuelos aterrizan con adelanto**. Esto se debe a que las aerolíneas suelen incluir un margen de holgura en los tiempos de vuelo programados (*block time*) para absorber pequeñas demoras operativas.

Esta particularidad condiciona directamente nuestro enfoque de modelado: **el modelo debe estar en capacidad de predecir valores negativos**. Por ende, no podemos aplicar transformaciones directas que asuman valores estrictamente positivos (como una transformación logarítmica estándar sobre `arr_delay`) ni utilizar modelos o métricas restringidos a variables no negativas sin una adaptación previa.

### 2.3 Tipos de variables

A continuación, inspeccionamos los tipos de datos de las 9 variables predictoras que utilizaremos en el modelado y de la variable objetivo. Omitimos las 26 columnas restantes ya que, como justificaremos en la sección 2.5, no pueden formar parte del espacio de características.

In [ ]:
tipos = pd.DataFrame({
    "dtype": df[PREDICTORAS + [OBJETIVO]].dtypes.astype(str),
    "rol": ["predictora"] * len(PREDICTORAS) + ["objetivo"],
})
tipos

Al analizar la naturaleza de estas variables encontramos una mezcla de tipos que requerirá un preprocesamiento cuidadoso:

1. **Variables categóricas nominales (`op_unique_carrier`, `origin`, `dest`):**
   Almacenan identificadores en texto (códigos de aerolínea y códigos IATA de aeropuertos). No poseen un orden intrínseco ni representan magnitudes, por lo que deberán codificarse numéricamente para ser consumidas por los algoritmos.

2. **Variables con estructura temporal y cíclica (`month`, `day_of_week`, `crs_dep_time`, `crs_arr_time`):**
   Aunque pandas las interpreta como enteros (`int64`), no son variables numéricas continuas convencionales:
   - `month` y `day_of_week` representan ciclos en el calendario: el domingo (7) es contiguo al lunes (1), y diciembre (12) precede inmediatamente a enero (1). Tratarlas de forma lineal sin considerar su periodicidad impondría una distancia artificial entre extremos.
   - `crs_dep_time` y `crs_arr_time` corresponden a horas en formato militar (`hhmm`, por ejemplo 1359 para 13:59). En esta representación, de las 13:59 (1359) a las 14:00 (1400) solo transcurre 1 minuto, pero la diferencia numérica entera es de 41 unidades. Además, la medianoche presenta una discontinuidad similar (de 2359 a 0001).

3. **Variables numéricas continuas estándar (`crs_elapsed_time`, `distance`):**
   Son magnitudes físicas y temporales reales (duración estimada en minutos y distancia de ruta en millas), donde la escala y distancia matemática sí son lineales y consistentes.

Identificar estas particularidades desde el inicio es clave, pues orienta las transformaciones de ingeniería de características y codificación que detallaremos en la sección 4.

### 2.4 Estadísticas de la variable objetivo

In [ ]:
stats = df[OBJETIVO].agg(["count", "mean", "median", "std", "min", "max"])
stats["p25"] = df[OBJETIVO].quantile(0.25)
stats["p75"] = df[OBJETIVO].quantile(0.75)
print(stats.round(2).to_string())

print(f"\nVuelos que llegan adelantados (arr_delay < 0): "
      f"{(df[OBJETIVO] < 0).mean() * 100:.1f} %")
print(f"Valores faltantes en {OBJETIVO}: "
      f"{df[OBJETIVO].isna().sum()} filas ({df[OBJETIVO].isna().mean() * 100:.2f} %)")

El resumen estadístico aporta información muy valiosa sobre el comportamiento real de los vuelos:

- **Predominio de llegadas a tiempo o anticipadas:**
  La **mediana es de −6 minutos** y aproximadamente el **60.9 %** de los vuelos aterriza antes de lo programado. En condiciones normales de operación, el vuelo típico de la muestra no experimenta retrasos significativos.
- **Fuerte asimetría positiva (hacia la derecha):**
  La **media (7.55 minutos)** se encuentra notablemente por encima de la mediana. Esta discrepancia se explica porque los adelantos están físicamente acotados (el registro mínimo es de −78 minutos), mientras que los retrasos pueden extenderse por horas debido a contingencias operativas (el valor máximo alcanza los 2 014 minutos, equivalente a más de 33 horas).
- **Dispersión y presencia de valores extremos:**
  El 50 % central de los vuelos se concentra en una franja estrecha de apenas 25 minutos (el rango intercuartílico va de −15 a +10 minutos). Sin embargo, la desviación estándar es de 55.8 minutos, reflejando el fuerte peso de la cola derecha.

De estas observaciones se desprenden dos decisiones metodológicas importantes:
1. **Selección de métricas robustas:** Como la media se ve fuertemente afectada por eventos extremos poco frecuentes, las métricas cuadráticas como el RMSE tenderán a sobrerreaccionar ante anomalías atípicas. Esto justifica que prioricemos el **Error Absoluto Medio (MAE)** como métrica principal de evaluación en la sección 9.
2. **Presencia de valores nulos:** Notamos que la variable objetivo cuenta con 9 836 registros válidos frente a las 10 000 filas totales. Los 164 registros faltantes (1.64 %) corresponden a vuelos que no completaron su ruta habitual, situación que analizamos a fondo en la sección 2.6.

*(El detalle visual y análisis gráfico de esta distribución se profundiza en la sección 3 de análisis exploratorio).*

### 2.5 Por qué se usan 9 columnas de las 35

El dataset original contiene 35 atributos, pero decidimos trabajar únicamente con 9 de ellos como variables predictoras. Esta exclusión de 26 columnas responde a un principio metodológico estricto: **evitar la fuga de información (*data leakage*) garantizando que el modelo utilice únicamente datos disponibles antes del despegue**.

Clasificamos las columnas descartadas según su motivo de exclusión:

- **Variables ocurridas durante o después del vuelo (Fuga directa):**
  Campos como `dep_time`, `dep_delay`, `taxi_out`, `wheels_off`, `wheels_on`, `taxi_in`, `arr_time`, `actual_elapsed_time` y `air_time` describen el desarrollo real del trayecto. Aunque variables como el retraso en la salida (`dep_delay`) correlacionan fuertemente con el retraso final, incluirlas produciría un modelo ficticio y sin utilidad práctica: cuando el usuario o la aerolínea necesitan anticipar el retraso para tomar decisiones operativas, el vuelo aún no ha salido y estos datos simplemente no existen.
- **Desglose de causas del retraso (Fuga retrospectiva):**
  Columnas como `carrier_delay`, `weather_delay`, `nas_delay`, `security_delay` y `late_aircraft_delay` corresponden a la auditoría posterior que realiza la aerolínea para justificar la demora una vez concluido el vuelo. Utilizarlas equivaldría a entrenar el modelo con la respuesta desglosada.
- **Estado final de la operación:**
  `cancelled`, `cancellation_code` y `diverted` registran cancelaciones o desvíos que se confirman durante la operación, por lo que tampoco están disponibles en la ventana temporal de predicción previa al despegue.
- **Variables redundantes o de baja utilidad predictiva:**
  - `year`: es un valor constante (2024) y carece de varianza.
  - `op_carrier_fl_num`: es un número de vuelo administrativo que no aporta un patrón generalizable.
  - `fl_date` y `day_of_month`: aportan alta dimensionalidad y su efecto temporal queda mejor capturado de forma periódica a través de `month` y `day_of_week`.
  - Nombres de ciudades y estados (`origin_city_name`, `origin_state_nm`, etc.): duplican la información espacial que ya proporcionan con mayor granularidad los códigos IATA `origin` y `dest`.

### 2.6 Limitaciones del conjunto de datos

Al explorar el conjunto identificamos varios retos y limitaciones inherentes al problema y a los datos disponibles:

1. **Ausencia de etiqueta en vuelos no completados (1.64 % de nulos en `arr_delay`):**
   Las 164 filas sin valor en la variable objetivo corresponden exactamente a 122 vuelos cancelados y 42 vuelos desviados. Al no haber aterrizado en su destino programado, no tienen un tiempo de llegada observable. En un esquema de aprendizaje supervisado para regresión, estas instancias no pueden utilizarse para entrenar ni evaluar el modelo, por lo que requerirán ser filtradas en la fase de preparación.
2. **Ausencia estructural de nulos en variables previas al vuelo:**
   Al evaluar las 9 predictoras seleccionadas encontramos un **0.00 % de valores faltantes**, una condición que se mantiene incluso al revisar los más de 7 millones de registros del dataset global. Esto ocurre porque la información del itinerario programado es un requisito administrativo que siempre se registra completo. Por el contrario, los valores nulos naturales del dataset se concentran exclusivamente en variables operativas posteriores (como tiempos de carreteo o cancelaciones), las cuales no podemos usar por fuga de información. Esta particularidad motiva la estrategia explicada en la sección 2.7.
3. **Alta cardinalidad en aeropuertos (`origin` y `dest`):**
   En la muestra encontramos 284 aeropuertos de origen y 287 de destino distintos frente a 15 aerolíneas. Aplicar una codificación tradicional de tipo *One-Hot Encoding* sobre todas estas categorías dispararía la dimensionalidad a casi 600 columnas dispersas para 10 000 observaciones, lo que aumentaría el riesgo de sobreajuste y costo computacional.
4. **Falta de información contextual exógena (clima y rotación de aeronaves):**
   El dataset no cuenta con variables climáticas en tiempo real ni con el número de cola o matrícula del avión (*tail number*), lo que impide rastrear si la aeronave asignada venía con retraso acumulado de un trayecto previo (efecto dominó o *reactionary delay*). Esta ausencia de factores causales clave acota de forma natural el límite teórico de precisión (*R²* o MAE) que cualquier modelo puede alcanzar con estas variables.
5. **Alcance y representatividad:**
   El estudio se enfoca en vuelos domésticos dentro de Estados Unidos durante el año 2024. Aunque la muestra cubre adecuadamente la estacionalidad mensual y semanal, los resultados y patrones encontrados no deben extrapolarse a rutas internacionales ni a otros periodos temporales con condiciones macroeconómicas o regulatorias distintas.

### 2.7 Inducción controlada de valores faltantes

> **Nota de transparencia metodológica:**
> Los valores faltantes que analizaremos y trataremos a continuación no estaban presentes originalmente en el dataset, sino que fueron inducidos de manera controlada y justificada para dar cumplimiento a los lineamientos del proyecto.

La guía del curso solicita implementar y evaluar una estrategia de imputación en al menos una variable predictora que contenga entre un 0.1 % y un 2.0 % de valores ausentes. Sin embargo, como evidenciamos en el análisis anterior, las 9 variables predictoras válidas (previas al despegue) presentan un 0.00 % de nulos en la totalidad de los datos, mientras que los nulos reales corresponden únicamente a variables post-vuelo descartadas por fuga de información.

Para conciliar el rigor contra la fuga de datos con los requerimientos prácticos del curso, optamos por **inducir un 1 % de valores faltantes (`NaN`) de forma controlada y reproducible sobre la variable `crs_elapsed_time`**.

Seleccionamos `crs_elapsed_time` (duración programada del vuelo) con base en los siguientes criterios:
1. **Naturaleza numérica continua:** Permite aplicar y validar directamente técnicas estadísticas de imputación (como la imputación por la mediana en el pipeline de preprocesamiento).
2. **Consistencia y redundancia contextual:** Al estar estrechamente ligada a la distancia del vuelo (`distance`) y a la diferencia entre los horarios programados, la imputación descansa sobre información implícita existente en las demás variables, minimizando distorsiones artificiales en el modelo.
3. **Plausibilidad en el negocio:** Es factible que en la práctica un vuelo recién programado o en código compartido (*codeshare*) no tenga consolidada de inmediato la duración estimada del itinerario, a diferencia de variables fijas e indispensables como el aeropuerto de origen o destino.

La inducción se realiza fijando la semilla global (`RANDOM_STATE = 42`), garantizando que las observaciones afectadas sean exactamente las mismas en cada corrida y preservando la completa reproducibilidad del experimento.

In [ ]:
# Faltantes INDUCIDOS: ninguna predictora sin fuga del dataset original
# tiene nulos (0.00 % sobre 7 079 081 filas). Ver la celda markdown anterior.
# Semilla fija -> el conjunto de filas afectadas es el mismo en cada ejecucion.
FRACCION_FALTANTES = 0.01

rng = np.random.default_rng(RANDOM_STATE)
idx_faltantes = rng.choice(
    df.index, size=int(len(df) * FRACCION_FALTANTES), replace=False
)
df.loc[idx_faltantes, "crs_elapsed_time"] = np.nan

pct_faltantes = df["crs_elapsed_time"].isna().mean() * 100
print(f"NaN inducidos en crs_elapsed_time: {len(idx_faltantes):,} filas")
print(f"Porcentaje resultante:             {pct_faltantes:.2f} %")
print(f"Dentro del rango exigido (0.1 %-2 %): {0.1 <= pct_faltantes <= 2.0}")

## 3. Exploración de datos (EDA)


## 4. Preparación de los datos


## 5. Separación train / test


## 6. Prevención de fuga de información


## 7. Modelo base (baseline)


## 8. Modelo predictivo


## 9. Evaluación


## 10. Interpretación de resultados


## 11. Serialización y verificación
